# B07 — Unified `/predict` Eval (Type 1 + Type 2)

Converts dataset samples into the unified request schema and sends them to `/predict`.

**Unified schema:**
```json
{
  "query_id": "T1_0001",
  "type": "type1",
  "query": "Is Student A eligible for graduation?",
  "premises": ["...", "..."],
  "options": ["Yes", "No", "Uncertain"]
}
```

- **Type 1** — `Logic_Based_Educational_Queries.json`; MCQ options embedded or as list; polar questions use `["Yes", "No", "Uncertain"]`.
- **Type 2** — `type2_physics_questions_NL_sample100.csv`; no premises/options; returns mock until type 2 is merged.
- Compares predicted answer against gold label and reports accuracy + latency.

In [1]:
import json, re, time, asyncio, statistics, random
from pathlib import Path
from collections import Counter

import httpx
import pandas as pd

API_BASE    = "https://api.iamphuckhang.dev"
# /predict returns the full response: answer + FOL of premises/question/options
# under `routing_diagnostics` (the 6 official submission fields are a subset).
PREDICT_URL = f"{API_BASE}/predict"

# --- sample sizes (random selection; set to None for full dataset) ---
N_TYPE1_MCQ = 25            # randomly drawn from MCQ type-1 questions
N_TYPE1_YNU = 25            # randomly drawn from polar/YNU type-1 questions
N_TYPE2     = 50            # randomly drawn from the type-2 set
SEED        = 42            # reproducible sampling
CONCURRENCY = 8
TIMEOUT     = 60.0

random.seed(SEED)

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src/exact/datasets/exact").exists():
    ROOT = ROOT.parent
DATA = ROOT / "src/exact/datasets/exact"
assert DATA.exists(), f"dataset dir not found from {Path.cwd()}"
print("dataset :", DATA)
print("endpoint:", PREDICT_URL)

dataset : /home/phuckhang/MyWorkspace/Exact2026/src/exact/datasets/exact
endpoint: https://api.iamphuckhang.dev/predict


## Build unified sample list

In [2]:
_OPTION_LINE = re.compile(r"^\s*([A-E])[.)]\s+(.+)$", re.MULTILINE)

def build_type1_samples(path: Path) -> list[dict]:
    """Build every type-1 sample (no truncation; sampling happens later)."""
    raw = json.load(open(path))
    samples = []
    for g_idx, group in enumerate(raw):
        for q_idx, (question, gold) in enumerate(zip(group["questions"], group["answers"])):
            is_mcq = bool(_OPTION_LINE.search(question))
            # Send no options. MCQ options are embedded in the question body
            # (A./A) lines) and extracted server-side; polar questions use the
            # native YNU path (check_ynu) returning Yes/No/Uncertain directly.
            # Supplying ["Yes","No","Uncertain"] would force is_mcq=True server-side.
            samples.append({
                "query_id": f"T1_{g_idx:04d}_{q_idx:02d}",
                "type": "type1",
                "query": question,
                "premises": group["premises-NL"],
                "options": None,
                "_gold": gold,
                "_is_mcq": is_mcq,
            })
    return samples

def build_type2_samples(path: Path) -> list[dict]:
    """Build every type-2 sample (no truncation; sampling happens later)."""
    df = pd.read_csv(path)
    samples = []
    for _, row in df.iterrows():
        samples.append({
            "query_id": str(row["id"]),
            "type": "type2",
            "query": str(row["question"]),
            "premises": None,
            "options": None,
            "_gold": str(row["answer"]),
            "_unit": str(row.get("unit", "")),
        })
    return samples

def _sample(pool: list[dict], n: int | None) -> list[dict]:
    """Random subset of `pool` (or the whole pool if n is None / >= len)."""
    if n is None or n >= len(pool):
        return list(pool)
    return random.sample(pool, n)

# --- type 1: split by question shape, then sample each stratum ---
t1_all = build_type1_samples(DATA / "Logic_Based_Educational_Queries.json")
t1_mcq_pool = [s for s in t1_all if s["_is_mcq"]]
t1_ynu_pool = [s for s in t1_all if not s["_is_mcq"]]
type1_samples = _sample(t1_mcq_pool, N_TYPE1_MCQ) + _sample(t1_ynu_pool, N_TYPE1_YNU)

# --- type 2: single random sample ---
t2_all = build_type2_samples(DATA / "type2_physics_questions_NL_sample100.csv")
type2_samples = _sample(t2_all, N_TYPE2)

all_samples = type1_samples + type2_samples

t1_mcq  = sum(1 for s in type1_samples if s["_is_mcq"])
t1_ynu  = len(type1_samples) - t1_mcq
print(f"Type 1 pool : MCQ {len(t1_mcq_pool)}, polar/YNU {len(t1_ynu_pool)}")
print(f"Type 2 pool : {len(t2_all)}")
print(f"Sampled (seed={SEED}):")
print(f"  Type 1 : {len(type1_samples)} samples  (MCQ: {t1_mcq}, polar/YNU: {t1_ynu})")
print(f"  Type 2 : {len(type2_samples)} samples")
print(f"  Total  : {len(all_samples)} samples")
print()
print("Example type1 payload:")
ex = {k: v for k, v in type1_samples[0].items() if not k.startswith("_")}
print(json.dumps(ex, indent=2)[:600])

Type 1 pool : MCQ 359, polar/YNU 449
Type 2 pool : 100
Sampled (seed=42):
  Type 1 : 50 samples  (MCQ: 25, polar/YNU: 25)
  Type 2 : 50 samples
  Total  : 100 samples

Example type1 payload:
{
  "query_id": "T1_0379_00",
  "type": "type1",
  "query": "Based on the above premises, which statement can be inferred if we know that H\u00e0 has accumulated 80 credits in the training program?\nA. H\u00e0 is eligible for an internship because she has accumulated 80 credits.\nB. H\u00e0 is not eligible for an internship due to insufficient credits.\nC. H\u00e0 must take remedial courses before applying for the internship.\nD. H\u00e0\u2019s internship application was submitted after June 1st.",
  "premises": [
    "Students must accumulate at least 65% of the total credits of their trainin


## Send to `/predict`

In [3]:
def to_payload(sample: dict) -> dict:
    """Convert a sample to the unified /predict request body."""
    body: dict = {
        "query_id": sample["query_id"],
        "type": sample["type"],
        "query": sample["query"],
    }
    if sample["premises"]:
        body["premises"] = sample["premises"]
    if sample["options"]:
        body["options"] = sample["options"]
    return body


async def call(client, sem, sample):
    async with sem:
        t0 = time.perf_counter()
        err, body = None, None
        try:
            r = await client.post(PREDICT_URL, json=to_payload(sample), timeout=TIMEOUT)
            r.raise_for_status()
            body = r.json()
        except Exception as e:
            err = repr(e)
    return {
        **sample,
        "_response": body,
        "_latency": time.perf_counter() - t0,
        "_error": err,
    }


async def run_eval(samples):
    sem = asyncio.Semaphore(CONCURRENCY)
    done = 0
    async with httpx.AsyncClient() as client:
        async def wrapped(s):
            nonlocal done
            res = await call(client, sem, s)
            done += 1
            if done % 5 == 0 or done == len(samples):
                print(f"  {done}/{len(samples)}", end="\r")
            return res
        return await asyncio.gather(*(wrapped(s) for s in samples))


print(f"Sending {len(all_samples)} requests (concurrency={CONCURRENCY})...")
t0 = time.perf_counter()
results = await run_eval(all_samples)
wall = time.perf_counter() - t0

errors  = [r for r in results if r["_error"]]
success = [r for r in results if not r["_error"]]
print(f"\nSuccess : {len(success)}/{len(results)}")
print(f"Errors  : {len(errors)}")
print(f"Wall    : {wall:.1f}s")

Sending 100 requests (concurrency=8)...
  100/100
Success : 82/100
Errors  : 18
Wall    : 384.1s


## Results per sample

In [4]:
for r in success:
    resp = r["_response"][0]
    pred = resp.get("answer", "?")
    gold = r["_gold"]
    ok   = "✓" if str(pred).strip().upper() == str(gold).strip().upper() else "✗"
    print(f"{ok} [{r['query_id']}] pred={pred!r:12s} gold={gold!r:10s}  "
          f"{resp.get('question_type','?'):6s}  {r['_latency']:.1f}s")
    if resp.get("error"):
        print(f"    !! {resp['error']}")

✗ [T1_0379_00] pred='A'          gold='B'         mcq     43.9s
✗ [T1_0077_00] pred='B'          gold='Unknown'   mcq     55.0s
✓ [T1_0012_00] pred='D'          gold='D'         mcq     41.2s
✓ [T1_0178_00] pred='A'          gold='A'         mcq     65.1s
✓ [T1_0398_01] pred='A'          gold='A'         mcq     33.1s
✗ [T1_0332_00] pred='A'          gold='Unknown'   mcq     20.8s
✓ [T1_0355_00] pred='D'          gold='D'         mcq     50.1s
✗ [T1_0269_00] pred='C'          gold='Unknown'   mcq     39.2s
✓ [T1_0016_00] pred='B'          gold='B'         mcq     34.6s
✓ [T1_0015_00] pred='A'          gold='A'         mcq     37.7s
✗ [T1_0144_00] pred='B'          gold='Unknown'   mcq     43.0s
✗ [T1_0311_00] pred='B'          gold='Unknown'   mcq     47.8s
✓ [T1_0360_00] pred='B'          gold='B'         mcq     52.6s
✓ [T1_0013_00] pred='C'          gold='C'         mcq     47.8s
✓ [T1_0340_00] pred='A'          gold='A'         mcq     50.8s
✗ [T1_0134_00] pred='C'          gold='U

## Accuracy + metrics

In [5]:
t1_res = [r for r in success if r["type"] == "type1"]
t2_res = [r for r in success if r["type"] == "type2"]

def accuracy(rows):
    if not rows:
        return 0.0, 0, 0
    correct = sum(
        1 for r in rows
        if str(r["_response"][0].get("answer","")).strip().upper()
           == str(r["_gold"]).strip().upper()
    )
    return correct / len(rows), correct, len(rows)

# --- Type 1 ---
acc1, c1, n1 = accuracy(t1_res)
t1_mcq_res = [r for r in t1_res if r["_is_mcq"]]
t1_ynu_res = [r for r in t1_res if not r["_is_mcq"]]
acc1_mcq, c1_mcq, n1_mcq = accuracy(t1_mcq_res)
acc1_ynu, c1_ynu, n1_ynu = accuracy(t1_ynu_res)

print("=== Type 1 Accuracy ===")
print(f"  Overall : {acc1:.1%}  ({c1}/{n1})")
print(f"  MCQ     : {acc1_mcq:.1%}  ({c1_mcq}/{n1_mcq})")
print(f"  Polar   : {acc1_ynu:.1%}  ({c1_ynu}/{n1_ynu})")

# --- Type 1 answer distribution ---
ans_dist = Counter(r["_response"][0].get("answer") for r in t1_res)
print("\n=== Type 1 answer distribution ===")
for k, v in ans_dist.most_common():
    print(f"  {str(k):12s}: {v}")

# --- Type 1 solver diagnostics ---
solver_used = sum(
    1 for r in t1_res
    if r["_response"][0].get("routing_diagnostics", {}).get("solver_used")
)
print(f"\n=== Type 1 solver ===\n  solver_used: {solver_used}/{n1}")

# --- Type 2 (mock) ---
print(f"\n=== Type 2 (mock) ===")
print(f"  Responses: {len(t2_res)}  (all return mock 'Unknown' until pipeline is merged)")

# --- Latency ---
lat = [r["_latency"] for r in success]
print(f"\n=== Latency (all) ===")
print(f"  mean={statistics.mean(lat):.2f}s  p50={statistics.median(lat):.2f}s  max={max(lat):.2f}s")

=== Type 1 Accuracy ===
  Overall : 55.9%  (19/34)
  MCQ     : 58.8%  (10/17)
  Polar   : 52.9%  (9/17)

=== Type 1 answer distribution ===
  Yes         : 14
  A           : 6
  B           : 5
  D           : 3
  C           : 3
  No          : 2
  Uncertain   : 1


AttributeError: 'NoneType' object has no attribute 'get'

In [6]:
# --- Diagnostic table: where does each type1 sample stop? -------------------
# Separates real logical uncertainty (Z3_TRUE_UNCERTAIN) from parser/verifier/
# mode gaps so we know what to fix next.
hdr = f"{'sample':14s} {'used':5s} {'verif':5s} {'supp':5s} {'mode':16s} {'opt_fol':7s} {'cause'}"
print(hdr)
print("-" * len(hdr))
from collections import Counter as _C
cause_counts = _C()
for r in [x for x in success if x["type"] == "type1"]:
    rd = r["_response"][0].get("routing_diagnostics") or {}
    qs = rd.get("query_spec") or {}
    n_opt_fol = sum(1 for c in qs.get("option_claims", []) if c.get("fol"))
    cause = rd.get("uncertainty_cause")
    cause_counts[cause] += 1
    print(f"{r['query_id']:14s} "
          f"{str(rd.get('solver_used')):5s} "
          f"{str(rd.get('premise_bundle_verified')):5s} "
          f"{str(qs.get('supported')):5s} "
          f"{str(qs.get('solver_mode')):16s} "
          f"{n_opt_fol:<7d} "
          f"{cause if cause is not None else '— (solved)'}")

print("\n=== uncertainty_cause distribution ===")
for c, n in cause_counts.most_common():
    print(f"  {str(c) if c is not None else '— (solved)':40s}: {n}")

# Premise warnings actually seen (non-blocking; solver still ran)
warn = _C()
for r in [x for x in success if x["type"] == "type1"]:
    for w in (r["_response"][0].get("routing_diagnostics") or {}).get("premise_warnings", []):
        warn[w.split(":")[0]] += 1
if warn:
    print("\n=== premise warnings (non-blocking) ===")
    for w, n in warn.most_common():
        print(f"  {w:40s}: {n}")

sample         used  verif supp  mode             opt_fol cause
---------------------------------------------------------------
T1_0379_00     True  True  True  entailment       4       — (solved)
T1_0077_00     True  True  True  entailment       4       — (solved)
T1_0012_00     True  True  True  entailment       4       — (solved)
T1_0178_00     True  True  True  entailment       4       — (solved)
T1_0398_01     True  True  True  entailment       4       — (solved)
T1_0332_00     False True  False entailment       0       — (solved)
T1_0355_00     True  True  True  entailment       4       — (solved)
T1_0269_00     True  True  True  entailment       4       — (solved)
T1_0016_00     True  True  True  entailment       4       — (solved)
T1_0015_00     True  True  True  entailment       4       — (solved)
T1_0144_00     True  True  True  entailment       4       — (solved)
T1_0311_00     True  True  True  entailment       4       — (solved)
T1_0360_00     True  True  True  ynu_mapped 

## Notes
- Sampling is random and stratified: `N_TYPE1_MCQ` MCQs + `N_TYPE1_YNU` polar/YNU type-1 questions, plus `N_TYPE2` type-2 questions. Set any of them to `None` to use the whole stratum. `SEED` controls reproducibility.
- Type 1 polar questions send no options so the server uses the native YNU path (`check_ynu`) returning Yes/No/Uncertain directly. MCQ options stay embedded in the question body and are extracted server-side.
- Type 2 returns a mock response (`answer="Unknown"`) until the physics pipeline is merged.
- `solver_used=False` means premise verification failed or the question was unsupported — answer is the uncertain token.
- To inspect one result: `next(r for r in results if r['query_id'] == 'T1_0000_00')['_response']`.

## FOL translations (premises / question / options)

`/predict` returns the full response, so the FOL is carried inside each result's
`routing_diagnostics` — no extra API calls:

- **premises → FOL**: `routing_diagnostics.parsed_premises[]` (`original_text` → `fol`)
- **translated question**: `routing_diagnostics.query_spec.main_claim_text` → `main_claim_fol`
- **options → FOL**: `routing_diagnostics.query_spec.option_claims[]` (`label, role, claim_text` → `fol`)

In [ ]:
# FOL comes straight from the /predict response already in `success`.
t1_fol = [r for r in success if r["type"] == "type1" and r["premises"]]
have_fol = sum(
    1 for r in t1_fol
    if (r["_response"][0].get("routing_diagnostics") or {}).get("parsed_premises")
)
print(f"Type 1 samples: {len(t1_fol)}  |  carrying FOL detail: {have_fol}")

In [ ]:
for r in t1_fol:
    resp = r["_response"][0]
    rd = resp.get("routing_diagnostics") or {}
    qs = rd.get("query_spec") or {}
    print("=" * 88)
    print(f"[{r['query_id']}]  pred={resp.get('answer')!r}  gold={r['_gold']!r}  "
          f"({'MCQ' if r['_is_mcq'] else 'polar/YNU'})")

    # --- premises -> FOL ---
    print("\n  PREMISES → FOL")
    for p in rd.get("parsed_premises", []):
        print(f"    • {p['original_text']}")
        print(f"        ⇒ {p['fol']}")
    if rd.get("premise_bundle_verified") is False:
        print(f"    [verification issues: {rd.get('premise_verification_issues')}]")

    # --- translated question ---
    print("\n  QUESTION → FOL")
    print(f"    text : {qs.get('main_claim_text')}")
    print(f"    fol  : {qs.get('main_claim_fol')}")
    print(f"    mode : {qs.get('question_format')}/{qs.get('solver_mode')}  "
          f"supported={qs.get('supported')}  negate={qs.get('negate_claim')}")

    # --- options -> FOL ---
    oc = qs.get("option_claims") or []
    if oc:
        print("\n  OPTIONS → FOL")
        for c in oc:
            text = c.get("claim_text") or c.get("normalized_text")
            print(f"    {c['label']}. [{c.get('role')}] {text}")
            print(f"        ⇒ {c.get('fol')}")
    if qs.get("issues"):
        print(f"\n  [query issues: {qs['issues']}]")
    print()